# 04 — Matrice de Décision Produit

**Livrable final du POC.** Ce notebook synthétise tous les résultats en une matrice
opérationnelle : *quel modèle déployer pour quel profil de club ?*

La matrice combine :
- Les **métriques quantitatives** (F1, latence, coût)
- Les **observations qualitatives** de l'analyse d'erreurs
- Les **contraintes opérationnelles** (données disponibles, budget, GPU)

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.evaluation.results import load_all_runs, aggregate_results

TEMPLATE = 'plotly_white'
MODEL_COLORS = {
    'tfidf_lr':                '#636EFA',
    'camembert':               '#EF553B',
    'setfit_camembert':        '#00CC96',
    'setfit_modernbert_en':    '#AB63FA',
    'setfit_modernbert_ml':    '#FFA15A',
    'setfit_mmbert':           '#FF97FF',
    'ministral':               '#19D3F3',
    'hybrid_setfit_ministral': '#FF6692',
}
MODEL_LABELS = {
    'tfidf_lr':                'TF-IDF + LR',
    'camembert':               'CamemBERT',
    'setfit_camembert':        'SetFit (mpnet-ml)',
    'setfit_modernbert_en':    'SetFit (ModernBERT-EN)',
    'setfit_modernbert_ml':    'SetFit (ModernBERT-ML)',
    'setfit_mmbert':           'SetFit (mmBERT)',
    'ministral':               'Ministral 8B',
    'hybrid_setfit_ministral': 'Hybride SetFit+Ministral',
}

In [2]:
df = load_all_runs()
df['model_label'] = df['model'].map(MODEL_LABELS).fillna(df['model'])
print(f"{len(df)} runs chargés")

383 runs chargés


---
## 1. Graphique spider/radar — profil de chaque stratégie

Chaque dimension représente une qualité importante pour le déploiement.

In [3]:
# Scores normalisés [0-1] pour chaque dimension (à remplir avec les résultats réels)
# Format : {'modele': [f1_fewshot, f1_fulldata, vitesse, cout_inv, simplicite, data_efficiency]}
# - f1_fewshot : F1 en 8-shot (normalisé)
# - f1_fulldata : F1 en full data
# - vitesse : 1 - latence normalisée (plus = plus rapide)
# - cout_inv : 1 - coût normalisé (plus = moins cher)
# - simplicite : facilité de déploiement
# - data_efficiency : capacité à apprendre avec peu de données

# Placeholder — à remplacer avec les vrais résultats une fois tous les scripts lancés
categories_radar = ['F1 few-shot (8)', 'F1 full data', 'Vitesse', 'Coût (inv.)', 'Simplicité', 'Data efficiency']

# Exemple de données (à mettre à jour)
profiles = {
    'TF-IDF + LR':         [0.50, 0.80, 1.00, 1.00, 1.00, 0.40],
    'CamemBERT':           [0.20, 0.99, 0.70, 0.90, 0.60, 0.20],
    'SetFit (mpnet-ml)':   [0.75, 0.92, 0.80, 0.95, 0.70, 0.85],
    'Ministral 8B':        [0.85, 0.85, 0.30, 0.40, 0.95, 1.00],
    'Hybride':             [0.82, 0.93, 0.65, 0.75, 0.50, 0.85],
}

fig = go.Figure()
colors = ['#636EFA', '#EF553B', '#00CC96', '#19D3F3', '#FF6692']

for (name, values), color in zip(profiles.items(), colors):
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],
        theta=categories_radar + [categories_radar[0]],
        fill='toself', fillcolor=color, opacity=0.15,
        line=dict(color=color, width=2),
        name=name,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1], tickformat='.0%')),
    title='Profil multidimensionnel des stratégies<br><sup>(placeholder — à mettre à jour avec les résultats réels)</sup>',
    height=550,
    template=TEMPLATE,
    showlegend=True,
)
fig.show()

---
## 2. Matrice de décision produit

In [4]:
# Extraire les résultats réels quand disponibles
f1_8shot = (
    df[df['regime'] == '8']
    .groupby(['model', 'dataset'])['f1_macro'].mean()
    .unstack(fill_value=float('nan'))
) if len(df) > 0 else pd.DataFrame()

f1_full = (
    df[df['regime'] == 'full']
    .groupby(['model', 'dataset'])['f1_macro'].mean()
    .unstack(fill_value=float('nan'))
) if len(df) > 0 else pd.DataFrame()

# Matrice de décision (valeurs à mettre à jour avec les résultats)
matrix = pd.DataFrame([
    {
        'Profil de club': '🆕 Club entrant',
        'Données disponibles': 'Aucune (0 mail étiqueté)',
        'Stratégie recommandée': 'Ministral 8B zero-shot',
        'F1 emails (attendu)': f1_8shot.get('emails', {}).get('ministral', '~0.80'),
        'Coût/mail': '~$0.000022',
        'Latence': '~1s',
        'GPU requis': '❌',
        'Remarque': 'Déploiement immédiat, sans données'
    },
    {
        'Profil de club': '📈 Club en onboarding',
        'Données disponibles': 'Peu (8-30 mails/cat.)',
        'Stratégie recommandée': 'SetFit few-shot',
        'F1 emails (attendu)': f1_8shot.get('emails', {}).get('setfit_camembert', '~0.90'),
        'Coût/mail': '$0',
        'Latence': '<50ms',
        'GPU requis': '⚡ Recommandé',
        'Remarque': 'Entraînable en <2 min sur CPU'
    },
    {
        'Profil de club': '🏆 Club établi',
        'Données disponibles': 'Beaucoup (>100 mails/cat.)',
        'Stratégie recommandée': 'CamemBERT fine-tuné',
        'F1 emails (attendu)': f1_full.get('emails', {}).get('camembert', '~0.99'),
        'Coût/mail': '$0',
        'Latence': '<100ms',
        'GPU requis': '✅ Nécessaire (entraînement)',
        'Remarque': 'Meilleure qualité avec données suffisantes'
    },
    {
        'Profil de club': '⚖️ Club exigeant',
        'Données disponibles': 'Variable',
        'Stratégie recommandée': 'Hybride SetFit + Ministral',
        'F1 emails (attendu)': '> SetFit seul',
        'Coût/mail': 'Variable (τ = levier)',
        'Latence': 'Variable',
        'GPU requis': '⚡ Recommandé',
        'Remarque': 'τ ajustable selon budget/qualité'
    },
])

matrix

,Profil de club,Données disponibles,Stratégie recommandée,F1 emails (attendu),Coût/mail,Latence,GPU requis,Remarque
0,🆕 Club entrant,Aucune (0 mail étiqueté),Ministral 8B zero-shot,~0.80,~$0.000022,~1s,❌,"Déploiement immédiat, sans données"
1,📈 Club en onboarding,Peu (8-30 mails/cat.),SetFit few-shot,0.947267,$0,<50ms,⚡ Recommandé,Entraînable en <2 min sur CPU
2,🏆 Club établi,Beaucoup (>100 mails/cat.),CamemBERT fine-tuné,0.986109,$0,<100ms,✅ Nécessaire (entraînement),Meilleure qualité avec données suffisantes
3,⚖️ Club exigeant,Variable,Hybride SetFit + Ministral,> SetFit seul,Variable (τ = levier),Variable,⚡ Recommandé,τ ajustable selon budget/qualité


---
## 3. Trade-off qualité vs coût — vue finale

In [5]:
# Vue synthétique : F1 full data (emails) vs coût d'inférence par mail.
# Taille des bulles = latence d'inférence (ms/mail).

# F1 full data par modèle sur emails
f1_dict = {}
lat_dict = {}
for model in df[df['regime'] == 'full']['model'].unique():
    sub = df[(df['regime'] == 'full') & (df['model'] == model) & (df['dataset'] == 'emails')]
    if not sub.empty:
        f1_dict[model] = sub['f1_macro'].mean()
        if 'inference_time_ms' in sub.columns:
            lat_dict[model] = sub['inference_time_ms'].mean()

rows = []
for m, f1 in f1_dict.items():
    lat = lat_dict.get(m)
    rows.append({
        'Modèle': MODEL_LABELS.get(m, m),
        'f1': f1,
        'cout_mail_usd': 0.0,            # modèles locaux = gratuits à l'inférence
        'latence_ms': lat if lat and lat == lat else 10.0,  # défaut si NaN/absent
    })

# Ajouter Ministral (zero-shot) : coût API non nul + latence réseau
m_data = df[(df['model'] == 'ministral') & (df['dataset'] == 'emails') & (df['regime'] == 'zero_shot')]
if not m_data.empty:
    lat = m_data['inference_time_ms'].mean() if 'inference_time_ms' in m_data.columns else 800
    rows.append({
        'Modèle': 'Ministral 8B (zero-shot)',
        'f1': m_data['f1_macro'].mean(),
        'cout_mail_usd': 0.000022,
        'latence_ms': lat if lat and lat == lat else 800.0,
    })

tradeoff = pd.DataFrame(rows).dropna(subset=['f1'])
# Sécurité : aucune taille NaN/<=0 (Plotly refuse)
tradeoff['latence_ms'] = tradeoff['latence_ms'].fillna(10.0).clip(lower=1.0)

# Axe x : on décale les coûts à 0 vers une petite valeur pour l'échelle log
tradeoff['cout_affiche'] = tradeoff['cout_mail_usd'].replace(0.0, 1e-7)

fig = px.scatter(
    tradeoff, x='cout_affiche', y='f1',
    size='latence_ms', text='Modèle', color='Modèle',
    size_max=40, log_x=True,
    title='Trade-off final : qualité vs coût (emails, full data)<br>'
          '<sup>Taille des bulles = latence d\'inférence | coût 1e-7 = gratuit (local)</sup>',
    labels={'cout_affiche': 'Coût par mail (USD, échelle log)', 'f1': 'F1 macro'},
    template=TEMPLATE,
)
fig.update_traces(textposition='top center')
fig.update_yaxes(tickformat='.0%', range=[0, 1.05])
fig.update_layout(height=520, showlegend=False)
fig.add_annotation(
    text="⬅ Idéal : F1 max, coût min",
    xref='paper', yref='paper', x=0.01, y=0.99,
    showarrow=False, font=dict(size=12, color='green')
)
fig.show()

---
## 4. Recommandations finales

*(À rédiger après analyse complète des résultats)*

**Questions clés à répondre :**
1. SetFit avec 8 exemples bat-il Ministral zero-shot sur les emails français ?
2. CamemBERT en full data justifie-t-il le coût d'entraînement vs TF-IDF ?
3. À quel seuil τ le système hybride offre-t-il le meilleur compromis ?
4. ModernBERT compense-t-il l'absence de spécialisation française dans SetFit ?

In [6]:
# Synthèse numérique des résultats clés
if len(df) > 0:
    print("=" * 60)
    print("SYNTHÈSE DES RÉSULTATS — EMAILS SPORTIFS")
    print("=" * 60)

    for regime_label, regime_val in [('8-shot', '8'), ('Full data', 'full'),
                                      ('Zero-shot', 'zero_shot')]:
        sub = df[(df['dataset'] == 'emails') & (df['regime'] == regime_val)]
        if sub.empty:
            continue
        best = sub.groupby('model')['f1_macro'].mean().nlargest(3)
        print(f"\n{regime_label} :")
        for model, f1 in best.items():
            label = MODEL_LABELS.get(model, model)
            print(f"  {label:35s} F1 = {f1:.1%}")

SYNTHÈSE DES RÉSULTATS — EMAILS SPORTIFS

8-shot :
  Hybride SetFit+Ministral            F1 = 95.0%
  SetFit (mpnet-ml)                   F1 = 94.7%
  SetFit (mmBERT)                     F1 = 92.9%

Full data :
  Hybride SetFit+Ministral            F1 = 99.3%
  TF-IDF + LR                         F1 = 99.2%
  SetFit (mpnet-ml)                   F1 = 99.2%

Zero-shot :
  Ministral 8B                        F1 = 79.3%
